# Global CDR calculations -- multi-schedule, Coiled cluster version

Combines `cdr-gcam-globalscaling-constApp-multiSchedule.ipynb` (reuse one SCEPTER run across multiple GCAM rock-application schedules) with `cdr-gcam-globalscaling-constApp-COILED.ipynb` (dispatch the per-combo sweep across a Coiled cluster instead of looping serially). Use this version when you're scaling to multiple schedules *and* have enough combos per schedule that the serial multiSchedule notebook is too slow.

Reuses a **single** set of SCEPTER outputs and scales it up against **multiple** GCAM rock-application schedules (`SCHEDULE_EXP_SETS`). This is only valid when the source SCEPTER run used a **fixed** application rate and the `repcells1` site-selection strategy -- under those conditions, both the climate forcing (`era5/gcam/create_clim_inputs_one-cell-per-region.ipynb`, `LOCATION_KEY = "repcells1"`) and the flat per-hectare application rate (`setup/dust/gcam/create-dust_temp-inputs.ipynb`, `USE_FIXED_APP_RATE = True`) are scenario-invariant, so one representative SCEPTER run can stand in for every GCAM scenario. What *does* differ by scenario is the GCAM regional rock-mass target and cropland-area trajectory (`dfrock_dict`/`dsag_dict`), so those are still read separately per schedule below, and the cluster sweep is dispatched once per schedule.


This script walks through how we scale up each region's representative run to calculate global CDR outcomes.

**Note:** This same workflow works for the `constApp` cases where cropland is capped at 100 percent and application evolves to track it, and for the cases where it has no cap at all. That's because the cap is imposed by the mass of existing deployments exceeding the mass of GCAM application -- a condition that only emerges in the simulations where we have to increase the rock application to mitigate overshoot of the 100 percent cropland condition. If rock application is truly constant (no overshoot response) then the cap never comes into play.

There are a few different ways we can scale up rock application. All would require the total amount of rock spread in each region to reach whatever GCAM simulated. This script assumes a fixed application rate and solves for the area required to reach the GCAM target.

The representative run's application amount was constructed with the following rules:
1. Assume a fixed application flux for each region (in t/ha/yr) and solve for the area required to match the GCAM amount of rock for that region and year.
2. If the cropland area required exceeds the total area available, increase the application flux accordingly (holding the area constant at 100%).

## Postprocessing in a single region
Since more area can be added to the region's ERW over time, we need to start the new area from "square 1" (t=0). That way ERW that starts in 2040 doesn't implicitly carry pH buffering neutralization that's been ongoing since 2020.

We do this by creating one new "deployment" (or "application cohort") every year (some deployments will spread no rock if total application doesn't increase). Each deployment follows the CDR trajectory of the region's representative run and is assigned an area equal to the change in regional area available for ERW. We ensure that the total amount of rock spread in the region each year aligns with the GCAM requirement.

### Year = 0 calculation
We start with the year 0 condition, letting $j$ refer to year and $i$ to a deployment. Since we only have one deployment in year 0, we spread all of the rock GCAM prescribes to that region and year to a single deployment. Formally,

$$M_\text{D, i} = M_\text{gcam, j}$$

when $i$ and $j$ equal zero. We can then calculate the total area required for the year 0 deployment ($A_\text{D, i}$; ha) because we know the application flux ($R_\text{j}$; t/ha/yr) in year 0 (and every year):

$$A_{D, 0} = \frac{M_\text{D, 0}}{R_0}.$$

Similarly, CDR for year 0 is just the CDR for that single deployment.

### Year > 0 calculation
For each year after year 0, the new annual deployment depends on how much *new* rock is spread relative to what existing deployments are already going to do. For each deployment, $i$, we calculate the mass of rock spread in a year by:

$$M_\text{D, i, j} = R_\text{j} \times A_\text{D, i}$$

Then, we get the amount of rock spread by existing deployments ($\mu_\text{j}$; t/yr) by summing them:

$$\mu_\text{j} = \sum_\text{i=0}^n M_\text{D, i, j}$$

Next, we can calculate how much new rock needs to be spread in year $j$ by comparing what existing deployments will do to what GCAM requires ($M_\text{GCAM, j}$; t/yr):

$$M_\text{new, j} = M_\text{GCAM, j} - \mu_\text{j}.$$

This new rock mass will be spread across the year's new deployment, at an area determined by the deployment's application flux that year. Since new deployments always start at $R_0$, we get:

$$A_\text{D, i} = \frac{M_\text{new, j}}{R_0}.$$

Deployment areas are generally fixed through time, with one exception. Since we rely on a single representative run to simulate many ERW deployments evolving over time, it's possible to reach a condition where $M_\text{new, j}$ is negative. That's because new deployments always start at $R_0$, but the representative rock spreading timeseries was calculating implicitly assuming new area kicks in at $R_j$, which can exceed $R_0$ in regions where there isn't enough cropland to hit $M_\text{GCAM, j}$ if $R_j=R_0$. The lower $R_0$ of the new deployment, compared to $R_j$, leads to a rock deficit that needs to be made up by excess area. When that deployment finally hits $R_j$ later on, it'll have "too much" area and we might see $M_\text{new, j}$ become negative.

There's no clear physical meaning for a negative $M_\text{new, j}$. It's just a scaling quirk that we have to deal with that falls out of our decision to use a single representative run to capture the time-evolution of many emerging "deployments" through time. We expect that quirk to be pretty small, likely much smaller than the uncertainty associated with other model decisions like the specific surface area or CEC. Moreover, as long as GCAM deployments see rock amendments stabilizing for long enough after scale-up (and most do), deployment areas, too, will stabilize back so the total area equals the available cropland area.

Since the more recent deployments are typically responsible for the area (and, later, mass) overshoot, we trim area off those first (first in, first out). If they need to be totally trimmed, then they are "turned off" completely. If $M_\text{new, j}$ becomes positive later, we start with a new deployment rather than turning a trimmed one back on or increasing its area.

We could alternatively explore a last in, first out rule, or a "dormancy" rule where trimmed deployments can come back (picking up where they left off) when $M_\text{new, j}$ turns positive. Generally, we don't expect $M_\text{new, j}$ to become positive again after it first turns negative since all individual deployments see their $R$ staying the same or increasing monotonically.

### Calculating CDR
The area calculations are the tricky part. Once we have those, calculating CDR for a given region and year ($\text{CDR}_\text{j}$; t/yr) is simple. It's just the sum of every deployment's yearly CDR flux (t/ha/yr) multiplied by the deployment's area:

$$\text{CDR}_\text{j} = \sum_{i=0}^n \text{CDR}_\text{rate, i, j} \times A_\text{D, i}$$

### Variables
This postprocessing effort produces or uses the following variables, defined over the following coordinates:
| VAR                   | COORDS                     | DESCRIPTION                    | UNITS          |
|-----------------------|----------------------------|--------------------------------|----------------|
| $R$                   | time, region, deployment   | application flux               | t/ha/yr        |
| $A_\text{D}$          | time, region, deployment   | deployment area                | ha             |
| $A_\text{total, ERW}$ | time, region               | total area for ERW             | ha             |
| $M_\text{GCAM}$       | time, region               | total rock spread              | t              |
| $M_\text{D}$          | time, region, deployment   | rock spread in deployment      | t              |
| $\mu$                 | time, region               | rock spread in old deployments | t              |
| $M_\text{new}$        | time, region               | rock spread in new deployment  | t              |
| $CDR_\text{flux}$     | time, region, deployment   | deployment-specific CDR        | t/ha/yr        |
| $CDR_\text{rate}$     | time, region               | region-specific CDR            | t/yr           |
| $A_\text{frac}$       | time, region               | total ERW area over total cropland area | ha    |


In [1]:
# synthetic_matrix_schedules()[5:10]

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import os

# --- read in cdr calculation functions
from ew_workflows import cdr_fxns_postproc as cfp
from ew_workflows.cflx_proc import scepter_rco2
# ---
#
# ----------------------------------------------------------------
# [ FOCUS RUNS ]
# exe_central1p5 ; exe_techBehaviorChange1p5 ; exe_noWeathering1p5
# ----------------------------------------------------------------
#
# ************************************************
# --- [ SCEPTER OUTPUTS ]  (one run, reused for every schedule)
DUSTSP      = 'gbas2'   # just for RCO2 determination
EXP_GROUP   = "gcam_opr_full2_cec4"
EXP_SET     = 'exe_central1p5'
SAVE_GROUP  = "gcam"
idx = "001"
#
# --- [ GCAM ROCK-APPLICATION SCHEDULES ]
#
# SCHEDULE_EXP_SETS = ['synthetic_scurve_early', 'synthetic_scurve_mid', 'synthetic_scurve_late']
# SCHEDULE_EXP_SETS = ['synthetic_scurve_early_sameMax', 'synthetic_scurve_mid_sameMax', 'synthetic_scurve_late_sameMax']
# TARGET_MAX_MTCO2_1 = 1000.0
# TARGET_MAX_MTCO2_2 = 2000.0
# SCHEDULE_EXP_SETS = [
#     f'scurve_{TARGET_MAX_MTCO2_1/1e3}Gt_early_sameMax', f'scurve_{TARGET_MAX_MTCO2_1/1e3}Gt_mid_sameMax', f'scurve_{TARGET_MAX_MTCO2_1/1e3}Gt_late_sameMax',
#     f'scurve_{TARGET_MAX_MTCO2_2/1e3}Gt_early_sameMax', f'scurve_{TARGET_MAX_MTCO2_2/1e3}Gt_mid_sameMax', f'scurve_{TARGET_MAX_MTCO2_2/1e3}Gt_late_sameMax',
# ]
#
# --- [ or pull names from create-synthetic-s-curves.ipynb's manifest ]
SYNTHETIC_MATRIX_S3_DIR = ('s3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/'
                           'one-cell-per-region/synthetic-rateConstrained')
def synthetic_matrix_schedules(names=None):
    """Schedule names from manifest.csv (all by default)."""
    manifest = pd.read_csv(f'{SYNTHETIC_MATRIX_S3_DIR}/manifest.csv', index_col='schedule_name')
    if names is None:
        return manifest.index.tolist()
    missing = [name for name in names if name not in manifest.index]
    if missing:
        raise KeyError(f'requested schedule name(s) not found in {SYNTHETIC_MATRIX_S3_DIR}/manifest.csv: {missing}')
    return list(names)
#
#

# SCHEDULE_EXP_SETS = ['exe_central1p5',] #'exe_techBehaviorChange1p5', 'exe_noWeathering1p5']
SCHEDULE_EXP_SETS = synthetic_matrix_schedules()[12:]  # 18 in manifest, run in batches of 6

#
# --- read in time-transient and integrated data
datpath = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}/{EXP_GROUP}/{EXP_SET}_{idx}"
fn_cdr = "ds_cdr.zarr"                  # time-integrated data
fn_cdr_trans = "ds_cdr_transient.zarr"  # time-transient data
# read in
# .load() so scatter is a plain broadcast, not a dask graph run on the cluster
ds = xr.open_zarr(os.path.join(datpath, fn_cdr)).load()
ds_trans = xr.open_zarr(os.path.join(datpath, fn_cdr_trans)).load()

# DGSA collapses apprate_base (etc.) into "collapsed_inputs", so promote it to a coord
IS_DGSA = 'apprate_base' not in ds.coords
if IS_DGSA:
    ds = ds.set_coords('apprate_base')
    ds_trans = ds_trans.set_coords('apprate_base')

# extract apprates for reading in rock flux data
apprates_arr = np.unique(ds.apprate_base.values)

# ************************************************
# --- [ ROCK FLUX INPUTS + CROPLAND MAP ]  (per schedule)
RUN_FORMAT = 'one-cell-per-region'
DEG_RES = 1
rock_fn = f'annual_rockflx_densest-ag-region_{DEG_RES}deg.pkl'
ag_fn = f'cropland+gcamRegions_time_{DEG_RES}deg.zarr'

dfrock_dict_bySchedule = {}
dsag_dict_bySchedule = {}

for schedule in SCHEDULE_EXP_SETS:
    if IS_DGSA:
        # GCAM rock target and cropland map are apprate-invariant, so reuse the
        # reference pkl/zarr and rescale apprate-dependent columns analytically
        REF_APPRATE = 5.0  # build_dgsa_climatedirs' default
        ref_apprate_name = f'{REF_APPRATE}-tHaYr-appRate-FIXED'
        ref_gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{schedule}/{ref_apprate_name}'
        df_rock_ref = pd.read_pickle(os.path.join(ref_gcam_dir, rock_fn))
        # drop_encoding(): zarr v3 codec objects stalled client.scatter()
        dsag_ref = xr.open_zarr(os.path.join(ref_gcam_dir, ag_fn)).load().drop_encoding()

        dfrock_dict = {}
        dsag_dict = {}
        for apprate in apprates_arr:
            apprate_name = f'{float(apprate)}-tHaYr-appRate-FIXED'
            scale = float(apprate) / REF_APPRATE
            df_this = df_rock_ref.copy()
            # FIXED apprate: actual == effective rate, other columns scale linearly
            # (calculate-rock-fluxes-from-GCAM.ipynb Scenario 3)
            df_this['effective_rate_t_ha'] = float(apprate)
            df_this['rockflx_t_ha'] = float(apprate)
            df_this['rockflx_Mt_ha'] = df_this['rockflx_t_ha'] / 1e6
            df_this['rockflx_Mt'] = df_rock_ref['rockflx_Mt'] * scale
            df_this['cropland_area_required_ha'] = df_rock_ref['cropland_area_required_ha'] / scale
            df_this['cropland_fraction_required'] = df_rock_ref['cropland_fraction_required'] / scale
            df_this['cropland_fraction_effective'] = df_this['cropland_fraction_required']
            dfrock_dict[apprate_name] = df_this
            dsag_dict[apprate_name] = dsag_ref
    else:
        dfrock_dict = {}
        for apprate in apprates_arr:
            apprate_name = f'{apprate}-tHaYr-appRate-FIXED'
            gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{schedule}/{apprate_name}'
            # [ read in ]
            dfrock_dict[apprate_name] = pd.read_pickle(os.path.join(gcam_dir, rock_fn))

        # ************************************************
        # --- [ CROPLAND MAP ]
        # crop areas are apprate-invariant, so share one map across apprate keys
        ref_apprate_name = f'{apprates_arr[0]}-tHaYr-appRate-FIXED'
        ref_gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{schedule}/{ref_apprate_name}'
        dsag_ref = xr.open_zarr(os.path.join(ref_gcam_dir, ag_fn)).load().drop_encoding()
        dsag_dict = {f'{apprate}-tHaYr-appRate-FIXED': dsag_ref for apprate in apprates_arr}

    dfrock_dict_bySchedule[schedule] = dfrock_dict
    dsag_dict_bySchedule[schedule] = dsag_dict

In [3]:
# synthetic_matrix_schedules()[:6]
# synthetic_matrix_schedules()[6:12]
# synthetic_matrix_schedules()[12:]

## Spin up the Coiled cluster

In [4]:
# --- Coiled cluster for the per-combo sweep (reused for every schedule)
import coiled
from dask.distributed import Client

N_WORKERS = 40          # scale with number of combos
WORKER_VM_TYPES = ["m6i.large"]
# worker-sized scheduler drops the connection by the 2nd/3rd schedule
SCHEDULER_VM_TYPES = ["r6i.xlarge"]  # 4 vCPU / 32GB

cluster = coiled.Cluster(
    name="cdr-gcam-globalscaling-constApp-multiSchedule",
    n_workers=N_WORKERS,
    worker_vm_types=WORKER_VM_TYPES,
    scheduler_vm_types=SCHEDULER_VM_TYPES,
    region="us-west-2",
    spot_policy="on-demand",
    tags={"Project": "scepter"},
    # no container: package sync mirrors the local env (incl. editable ew_workflows)
)
client = Client(cluster)
client

[2026-09-12 13:36:46,890][INFO    ][coiled] Fetching latest package priorities...
[2026-09-12 13:36:46,891][INFO    ][coiled.package_sync] Resolving your local /Users/tylerkukla/Github/iam-rtm-compare/uv.lock Python environment...
[2026-09-12 13:36:47,007][INFO    ][coiled.package_sync] Scanning 239 python packages...
[2026-09-12 13:36:47,218][INFO    ][coiled] Running pip check...
[2026-09-12 13:36:47,462][INFO    ][coiled] Validating environment...
[2026-09-12 13:36:48,485][INFO    ][coiled] Creating wheel for ~/Github/ew-workflows/src...
[2026-09-12 13:36:48,566][INFO    ][coiled] Creating wheel for ew-workflows...
[2026-09-12 13:36:49,720][INFO    ][coiled] Uploading coiled_local_src...
[2026-09-12 13:36:50,735][INFO    ][coiled] Uploading ew-workflows...
[2026-09-12 13:36:51,641][INFO    ][coiled] Creating software environment...
[2026-09-12 13:36:52,903][INFO    ][coiled] Creating Cluster (name: cdr-gcam-globalscaling-constApp-multiSchedule, https://cloud.coiled.io/clusters/20415

<Client: 'tls://10.1.20.38:8786' processes=39 threads=78, memory=253.71 GiB>

## Set up postprocessing decisions

In [5]:
# ---- cdr var used for global calculations
cdr_var  = 'cdr_adv'
# ---- skip the initial numerical blip (known SCEPTER quirk) ----
t_start = 0.2        # [yr] blip resolves by ~0.1
# ---- set up the rco2 correction factor
RCO2_GCAM = 0.3      # [tCDR/tRock] cdr potential under perfect efficiency
RCO2_SCEPTER = scepter_rco2[DUSTSP]  # [tCDR/tRock] cdr potential under perfect efficiency
rco2_correction_factor = RCO2_GCAM / RCO2_SCEPTER
# ---- which data to start with (time-integrated or transient)
USE_INTEGRATED = True   # derive fluxes from integrated data (avoids transient aliasing)

In [6]:
rco2_correction_factor

1.0

In [7]:
# varlist_deploy          : (time, deployment) on the annual grid
# varlist_region          : area-scaled regional total
# varist_region_cumfrac   : cumulative rock-weighted regional mean (fractions)
# varlist_rco2_correction : scaled to GCAM's RCO2

_catbudget_vars = [
    'catbudget_gbas', 'catbudget_adv', 'catbudget_tflx',
    'catbudget_sic', 'catbudget_other', 'catbudget_residual',
]

varlist_deploy      = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
    'int_dust_ton_ha_yr', 'noncarbsld_source_ca', 'noncarbsld_source_mg', 'noncarbsld_source_k',
    'noncarbsld_source_na', 'total_dissolution',
] + _catbudget_vars

varlist_region      = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat',
] + _catbudget_vars

varist_region_cumfrac = [
    'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
]
# catbudget_* are not fractions

varlist_rco2_correction = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'noncarbsld_source_ca',
    'noncarbsld_source_mg', 'noncarbsld_source_k', 'noncarbsld_source_na',
] + _catbudget_vars
# catbudget terms scale with RCO2, so correct them too

## Run the postprocess (parallelized across the cluster)

In [8]:
# --- one combo per task (loop body of cfp.gcam_postprocess_all_dims_constApp)
def _process_one_combo(
    isel_dict, ds_trans, ds, dfrock_dict, dsag_dict,
    loop_dims, site_dim, region_col, apprate_key_fn,
    varlist_deploy, varlist_region, varlist_region_cumfrac,
    varlist_rco2_correction, rco2_correction_factor,
    cdr_var, t_start, use_integrated,
):
    from ew_workflows import cdr_fxns_postproc as cfp

    ds_rep_meta = ds_trans.isel(**isel_dict)
    region_name = str(ds_rep_meta[site_dim].values)

    # fall back to the first key only if apprate_base is absent
    if 'apprate_base' in ds_rep_meta.coords:
        apprate_key = apprate_key_fn(float(ds_rep_meta['apprate_base'].values))
    else:
        apprate_key = next(iter(dfrock_dict))

    df_this     = dfrock_dict[apprate_key]
    region_rows = df_this[df_this[region_col] == region_name]
    if region_rows.empty:
        return None
    region_idx = int(region_rows['reg_id'].iloc[0])

    # optionally differentiate the integrated run
    if use_integrated:
        ds_rep = cfp._int_to_flux(ds.isel(**isel_dict))
    else:
        ds_rep = ds_rep_meta

    ds_out = cfp.gcam_postprocess_region_constApp(
        ds_rep                  = ds_rep,
        df_rock                 = dfrock_dict[apprate_key],
        dsag                    = dsag_dict[apprate_key],
        region_idx              = region_idx,
        varlist_deploy          = varlist_deploy,
        varlist_region          = varlist_region,
        varlist_region_cumfrac  = varlist_region_cumfrac,
        varlist_rco2_correction = varlist_rco2_correction,
        rco2_correction_factor  = rco2_correction_factor,
        cdr_var                 = cdr_var,
        t_start                 = t_start,
    )

    for dim in loop_dims:
        ds_out = ds_out.assign_coords({dim: ds_rep_meta[dim].values})
    ds_out = ds_out.expand_dims(loop_dims)

    return ds_out

In [9]:
# --- dispatch + gather every combo, once per schedule
import itertools
from distributed import progress

loop_dims  = [d for d in ds_trans.dims if d != 'time']
dim_ranges = [range(ds_trans.sizes[d]) for d in loop_dims]
isel_dicts = [dict(zip(loop_dims, idxs)) for idxs in itertools.product(*dim_ranges)]

# shared SCEPTER run, scattered once for all schedules
ds_trans_f = client.scatter(ds_trans, broadcast=True)
ds_f       = client.scatter(ds, broadcast=True)

results_raw_bySchedule = {}
for schedule in SCHEDULE_EXP_SETS:
    print(f'\n[{schedule}] dispatching {len(isel_dicts)} combos across the cluster...')

    # wrap in a list so scatter keeps each dict as one object
    dfrock_dict_f = client.scatter([dfrock_dict_bySchedule[schedule]], broadcast=True)[0]
    # no broadcast: fan-out to all workers hung (CommClosedError); workers fetch it peer-to-peer
    dsag_dict_f   = client.scatter([dsag_dict_bySchedule[schedule]])[0]

    futures = client.map(
        _process_one_combo,
        isel_dicts,
        ds_trans                = ds_trans_f,
        ds                       = ds_f,
        dfrock_dict              = dfrock_dict_f,
        dsag_dict                = dsag_dict_f,
        loop_dims                = loop_dims,
        site_dim                 = 'site',
        region_col               = 'region_nospaces',
        apprate_key_fn           = lambda v: f'{float(v)}-tHaYr-appRate-FIXED',
        varlist_deploy           = varlist_deploy,
        varlist_region           = varlist_region,
        varlist_region_cumfrac   = varist_region_cumfrac,
        varlist_rco2_correction  = varlist_rco2_correction,
        rco2_correction_factor   = rco2_correction_factor,
        cdr_var                  = cdr_var,
        t_start                  = t_start,
        use_integrated           = USE_INTEGRATED,
    )

    progress(futures)
    results_raw_bySchedule[schedule] = client.gather(futures)
    n_ok = sum(r is not None for r in results_raw_bySchedule[schedule])
    print(f'\n[{schedule}] {n_ok} / {len(isel_dicts)} combos returned data (rest skipped -- not in df_rock)')

    # release this schedule's futures (del, not cancel(), to spare the scheduler)
    del futures, dfrock_dict_f, dsag_dict_f


[cdrfyi-deliveries_k_gcam_L_gcam] dispatching 2304 combos across the cluster...

[cdrfyi-deliveries_k_gcam_L_gcam] 2304 / 2304 combos returned data (rest skipped -- not in df_rock)

[cdrfyi-deliveries_k_gcam_L2.0Gt] dispatching 2304 combos across the cluster...

[cdrfyi-deliveries_k_gcam_L2.0Gt] 2304 / 2304 combos returned data (rest skipped -- not in df_rock)

[cdrfyi-deliveries_k_gcam_L0.5Gt] dispatching 2304 combos across the cluster...

[cdrfyi-deliveries_k_gcam_L0.5Gt] 2304 / 2304 combos returned data (rest skipped -- not in df_rock)

[cdrfyi-deliveries_k_nemetMedian_L_gcam] dispatching 2304 combos across the cluster...

[cdrfyi-deliveries_k_nemetMedian_L_gcam] 2304 / 2304 combos returned data (rest skipped -- not in df_rock)

[cdrfyi-deliveries_k_nemetMedian_L2.0Gt] dispatching 2304 combos across the cluster...

[cdrfyi-deliveries_k_nemetMedian_L2.0Gt] 2304 / 2304 combos returned data (rest skipped -- not in df_rock)

[cdrfyi-deliveries_k_nemetMedian_L0.5Gt] dispatching 2304 com

In [10]:
# --- assemble each schedule's results (own cell so a gather can be redone without resubmitting)
ds_all_bySchedule = {}

for schedule in SCHEDULE_EXP_SETS:
    results = [r for r in results_raw_bySchedule[schedule] if r is not None]
    ds_all = xr.combine_by_coords(results, combine_attrs='override')

    # same derived globals that gcam_postprocess_all_dims_constApp adds at the end
    ds_all = ds_all.assign({
        'gcam_cdr_globe': ds_all['gcam_cdr'].sum(dim='site').assign_attrs(
            units='tCO2/yr', description='Global GCAM CDR (sum across active regions)'
        ),
        f'{cdr_var}_globe': ds_all[f'{cdr_var}_region'].sum(dim='site').assign_attrs(
            units='t/yr', description=f'Global modeled CDR ({cdr_var}), sum across active regions'
        ),
        'M_D_region': ds_all['M_D'].sum(dim='deployment').assign_attrs(
            units='t/yr', description='Total rock spread per region per year (sum of M_D across deployments)'
        ),
    })

    ds_all_bySchedule[schedule] = ds_all
    print(f'\n[{schedule}] Active sites:', list(ds_all.site.values))
    print(f'[{schedule}] Output dims:', dict(ds_all.sizes))

ds_all_bySchedule[SCHEDULE_EXP_SETS[0]]


[cdrfyi-deliveries_k_gcam_L_gcam] Active sites: [np.str_('Africa_Eastern'), np.str_('Africa_Northern'), np.str_('Africa_Southern'), np.str_('Africa_Western'), np.str_('Argentina'), np.str_('Australia_NZ'), np.str_('Brazil'), np.str_('Canada'), np.str_('Central_America_and_Caribbean'), np.str_('Central_Asia'), np.str_('China'), np.str_('Colombia'), np.str_('EU-12'), np.str_('EU-15'), np.str_('Europe_Eastern'), np.str_('Europe_Non_EU'), np.str_('European_Free_Trade_Association'), np.str_('India'), np.str_('Indonesia'), np.str_('Japan'), np.str_('Mexico'), np.str_('Middle_East'), np.str_('Pakistan'), np.str_('Russia'), np.str_('South_Africa'), np.str_('South_America_Northern'), np.str_('South_America_Southern'), np.str_('South_Asia'), np.str_('South_Korea'), np.str_('Southeast_Asia'), np.str_('Taiwan'), np.str_('USA')]
[cdrfyi-deliveries_k_gcam_L_gcam] Output dims: {'site': 32, 'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'time': 80, 'deployment': 80}

[cdrfyi-deliveries_k_gcam_L2.0

<xarray.Dataset> Size: 4GB
Dimensions:                                     (site: 32, apprate_base: 9,
                                                 dustrad: 8, cec_factor: 1,
                                                 time: 80, deployment: 80)
Coordinates:
  * site                                        (site) <U31 4kB 'Africa_Easte...
  * apprate_base                                (apprate_base) float64 72B 0....
  * dustrad                                     (dustrad) float64 64B 0.2 ......
  * cec_factor                                  (cec_factor) float64 8B 4.0
  * time                                        (time) float64 640B 0.0 ... 79.0
  * deployment                                  (deployment) int64 640B 0 ... 79
Data variables: (12/65)
    cdr_adv_deployment                          (site, apprate_base, dustrad, cec_factor, time, deployment) float64 118MB ...
    rockflx_t_ha_yr                             (site, apprate_base, dustrad, cec_factor, time, deployment) float64 118MB ...
    rco2_correction_factor                      (site, apprate_base, dustrad, cec_factor) float64 18kB ...
    cdr_dif_deployment                          (site, apprate_base, dustrad, cec_factor, time, deployment) float64 118MB ...
    cdr_dif_component_deployment                (site, apprate_base, dustrad, cec_factor, time, deployment) float64 118MB ...
    cdr_resp_component_deployment               (site, apprate_base, dustrad, cec_factor, time, deployment) float64 118MB ...
    ...                                          ...
    total_dissolution_region                    (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    int_dust_ton_ha_yr_region                   (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    fraction_total_dissolved_fromIntFlx_region  (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    gcam_cdr_globe                              (apprate_base, dustrad, cec_factor, time) float64 46kB ...
    cdr_adv_globe                               (apprate_base, dustrad, cec_factor, time) float64 46kB ...
    M_D_region                                  (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
Attributes:
    region:   Africa_Eastern

In [11]:
# --- reattach swept DGSA parameters as plain variables (not coordinates)
# names parsed from the "col1=val1 | col2=val2 | ..." collapsed_inputs string
if IS_DGSA:
    sample_combo = str(ds['collapsed_inputs'].values[0])
    param_cols = [kv.split('=')[0].strip() for kv in sample_combo.split('|')]
    param_cols = [c for c in param_cols if c in ds.variables]
    ds_params = ds[param_cols].reset_coords()  # demote e.g. apprate_base back to a variable
    # strip zarr v3 encoding (breaks the zarr_format=2 save)
    ds_params = ds_params.drop_encoding()
    for schedule in SCHEDULE_EXP_SETS:
        ds_all_bySchedule[schedule] = ds_all_bySchedule[schedule].merge(ds_params, join='left')

# Separate out just the regional / global data
Just to save a smaller version of the file

In [12]:
ds_sum_bySchedule = {schedule: ds_all.drop_dims("deployment") for schedule, ds_all in ds_all_bySchedule.items()}
ds_sum_bySchedule[SCHEDULE_EXP_SETS[0]]

<xarray.Dataset> Size: 44MB
Dimensions:                                     (site: 32, apprate_base: 9,
                                                 dustrad: 8, cec_factor: 1,
                                                 time: 80)
Coordinates:
  * site                                        (site) <U31 4kB 'Africa_Easte...
  * apprate_base                                (apprate_base) float64 72B 0....
  * dustrad                                     (dustrad) float64 64B 0.2 ......
  * cec_factor                                  (cec_factor) float64 8B 4.0
  * time                                        (time) float64 640B 0.0 ... 79.0
Data variables: (12/34)
    rco2_correction_factor                      (site, apprate_base, dustrad, cec_factor) float64 18kB ...
    M_GCAM                                      (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    mu                                          (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    M_new                                       (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    A_total_ERW                                 (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    cropland_ha_total                           (site, apprate_base, dustrad, cec_factor) float64 18kB ...
    ...                                          ...
    total_dissolution_region                    (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    int_dust_ton_ha_yr_region                   (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    fraction_total_dissolved_fromIntFlx_region  (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    gcam_cdr_globe                              (apprate_base, dustrad, cec_factor, time) float64 46kB ...
    cdr_adv_globe                               (apprate_base, dustrad, cec_factor, time) float64 46kB ...
    M_D_region                                  (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
Attributes:
    region:   Africa_Eastern

## Extra step: application -> dissolution -> CDR lag (cation-budget basis)

Optional diagnostic: for each schedule and combination of the swept dims, computes how many years elapse between rock **application**, rock **dissolution**, and **realized CDR** (cation-budget basis), per region and globally.

- `lag_ds` -- per-region lag, dims `(site, ...swept dims...)`.
- `global_lag_ds` -- global lag, dims `(metric, ...swept dims...)`, with `'global_curve_summed'` / `'cdr_weighted_regional_avg'` as variables.

Runs on `ds_sum`, not `ds_all` -- the lag calculation only needs region-level variables (`catbudget_*_region`, `M_GCAM`), not the `(time, deployment)` arrays.

In [13]:
# --- [ EXTRA STEP: application -> dissolution -> CDR lag, cation-budget basis ]
RCO2 = RCO2_GCAM   # [tCO2/tRock]
PERCENTILES = [0.25, 0.50, 0.75]

# one combo per task (loop body of cfp.compute_lag_table_all_dims)
def _process_one_lag_combo(sel_kwargs, dsall, RCO2, percentiles, site_dim):
    from ew_workflows import cdr_fxns_postproc as cfp
    import xarray as xr

    lag_df, region_curves = cfp.compute_lag_table(
        dsall, RCO2=RCO2, percentiles=percentiles, site_dim=site_dim, **sel_kwargs
    )
    global_lag_df = cfp.compute_global_lag(region_curves, lag_df, percentiles=percentiles)

    lag_piece    = xr.Dataset.from_dataframe(lag_df)
    global_piece = xr.Dataset.from_dataframe(global_lag_df)
    loop_dims = list(sel_kwargs.keys())
    for dim, val in sel_kwargs.items():
        lag_piece    = lag_piece.assign_coords({dim: val})
        global_piece = global_piece.assign_coords({dim: val})
    lag_piece    = lag_piece.expand_dims(loop_dims)
    global_piece = global_piece.expand_dims(loop_dims)
    return lag_piece, global_piece


for schedule in SCHEDULE_EXP_SETS:
    ds_sum_this = ds_sum_bySchedule[schedule]
    loop_dims_lag = [d for d in ds_sum_this.dims if d not in ('site', 'time', 'deployment')]
    combos = list(itertools.product(*[ds_sum_this[d].values for d in loop_dims_lag]))
    sel_kwargs_list = [dict(zip(loop_dims_lag, combo)) for combo in combos]
    print(f'\n[{schedule}] dispatching {len(sel_kwargs_list)} lag combos across the cluster...')

    ds_sum_f = client.scatter(ds_sum_this, broadcast=True)

    lag_futures = client.map(
        _process_one_lag_combo,
        sel_kwargs_list,
        dsall       = ds_sum_f,
        RCO2        = RCO2,
        percentiles = PERCENTILES,
        site_dim    = 'site',
    )
    progress(lag_futures)
    lag_results = client.gather(lag_futures)

    lag_pieces    = [r[0] for r in lag_results]
    global_pieces = [r[1] for r in lag_results]
    lag_ds        = xr.combine_by_coords(lag_pieces, combine_attrs='override')
    global_lag_ds = xr.combine_by_coords(global_pieces, combine_attrs='override')
    ds_sum_bySchedule[schedule] = ds_sum_this.merge(lag_ds).merge(global_lag_ds)

    print(f'[{schedule}] lag_ds dims:', dict(lag_ds.sizes))
    print(f'[{schedule}] global_lag_ds dims:', dict(global_lag_ds.sizes))

ds_sum_bySchedule[SCHEDULE_EXP_SETS[0]]


[cdrfyi-deliveries_k_gcam_L_gcam] dispatching 72 lag combos across the cluster...
[cdrfyi-deliveries_k_gcam_L_gcam] lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'site': 32}
[cdrfyi-deliveries_k_gcam_L_gcam] global_lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'metric': 15}

[cdrfyi-deliveries_k_gcam_L2.0Gt] dispatching 72 lag combos across the cluster...
[cdrfyi-deliveries_k_gcam_L2.0Gt] lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'site': 32}
[cdrfyi-deliveries_k_gcam_L2.0Gt] global_lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'metric': 15}

[cdrfyi-deliveries_k_gcam_L0.5Gt] dispatching 72 lag combos across the cluster...
[cdrfyi-deliveries_k_gcam_L0.5Gt] lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'site': 32}
[cdrfyi-deliveries_k_gcam_L0.5Gt] global_lag_ds dims: {'apprate_base': 9, 'dustrad': 8, 'cec_factor': 1, 'metric': 15}

[cdrfyi-deliveries_k_nemetMedian_L_gcam] dispatching 72 lag com

<xarray.Dataset> Size: 45MB
Dimensions:                                     (site: 32, apprate_base: 9,
                                                 dustrad: 8, cec_factor: 1,
                                                 time: 80, metric: 15)
Coordinates:
  * site                                        (site) <U31 4kB 'Africa_Easte...
  * apprate_base                                (apprate_base) float64 72B 0....
  * dustrad                                     (dustrad) float64 64B 0.2 ......
  * cec_factor                                  (cec_factor) float64 8B 4.0
  * time                                        (time) float64 640B 0.0 ... 79.0
  * metric                                      (metric) object 120B 't_app_c...
Data variables: (12/51)
    rco2_correction_factor                      (site, apprate_base, dustrad, cec_factor) float64 18kB ...
    M_GCAM                                      (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    mu                                          (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    M_new                                       (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    A_total_ERW                                 (site, apprate_base, dustrad, cec_factor, time) float64 1MB ...
    cropland_ha_total                           (site, apprate_base, dustrad, cec_factor) float64 18kB ...
    ...                                          ...
    t_diss_cdr_adv_region_P75                   (apprate_base, dustrad, cec_factor, site) float64 18kB ...
    t_app_diss_region_P75                       (apprate_base, dustrad, cec_factor, site) float64 18kB ...
    t_app_cdr_adv+sic_region_P75                (apprate_base, dustrad, cec_factor, site) float64 18kB ...
    t_diss_cdr_adv+sic_region_P75               (apprate_base, dustrad, cec_factor, site) float64 18kB ...
    global_curve_summed                         (apprate_base, dustrad, cec_factor, metric) float64 9kB ...
    cdr_weighted_regional_avg                   (apprate_base, dustrad, cec_factor, metric) float64 9kB ...
Attributes:
    region:   Africa_Eastern

## Save the result
One subdir per schedule under a sibling `*_multiSchedule` dir (not `datpath`).

In [14]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++
# --- save result (one subdir per schedule)
datpath_multiSchedule = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}/{EXP_GROUP}/{EXP_SET}_{idx}_multiSchedule"
SAVE_FN_ALL = "ds_cdr_globalScaling_constApp_ALL.zarr"
SAVE_FN_SUM = "ds_cdr_globalScaling_constApp_SUM.zarr"

for schedule in SCHEDULE_EXP_SETS:
    schedule_dir = os.path.join(datpath_multiSchedule, schedule)
    ds_all_bySchedule[schedule].to_zarr(os.path.join(schedule_dir, SAVE_FN_ALL), zarr_format=2, mode='w')
    ds_sum_bySchedule[schedule].to_zarr(os.path.join(schedule_dir, SAVE_FN_SUM), zarr_format=2, mode='w')

datpath_multiSchedule
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++

's3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_opr_full2_cec4/exe_central1p5_001_multiSchedule'

In [15]:
# --- tear down the cluster
client.close()
cluster.close()

[2026-09-12 14:16:14,406][INFO    ][coiled] Cluster 2041505 deleted successfully.


In [ ]:
# ---